<a href="https://colab.research.google.com/github/jdospina/jump-diffusion-estimation/blob/main/notebooks/jump_diffusion_playground_en.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎲 Jump Diffusion Playground

Welcome! This notebook lets you **play** with every jump-size distribution built into the [`jump-diffusion-estimation`](https://github.com/jdospina/jump-diffusion-estimation) library, in two ways:

1. **🎛 Free Play** — pick a jump distribution, wiggle its sliders, simulate a path, and watch Maximum Likelihood try to recover the parameters you chose.
2. **🎯 Guess the Parameters** — a mystery path is simulated with hidden, randomly-chosen parameters. Can you guess the distribution and its parameters just from looking at the path?

No local setup needed — everything below runs directly in Colab.

> Hay una versión en español de este notebook: `jump_diffusion_playground.ipynb`.

In [ ]:
# Install required dependencies
%pip install -q jump-diffusion-estimation ipywidgets

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

from jump_diffusion.simulation import JumpDiffusionSimulator
from jump_diffusion.estimation import JumpDiffusionEstimator
from jump_diffusion.distributions import (
    NormalJump,
    SkewNormalJump,
    SGEDJump,
    KouJump,
    StudentTJump,
)

plt.style.use('seaborn-v0_8')

## 🧬 The jump distributions

The library's `jump_distribution=` argument is *pluggable*: swap in any of these and the same simulator/estimator code works unchanged. Each one has its own parameters (`param_names`), sensible defaults, and optimization bounds -- we use those directly below to build the right sliders for whichever family you pick.

In [ ]:
DISTRIBUTIONS = {
    "Normal (Merton)": NormalJump(),
    "Skew-Normal": SkewNormalJump(),
    "SGED": SGEDJump(),
    "Kou (double-exp)": KouJump(),
    "Student-t": StudentTJump(),
}


def make_family_sliders(dist):
    """One FloatSlider per dist.param_names, ranged from param_bounds()
    (open bounds get a friendly spread around the default value).    """
    defaults = dist.default_params()
    bounds = dist.param_bounds()
    sliders = {}
    for name in dist.param_names:
        val = defaults[name]
        lo, hi = bounds[name]
        spread = 5 * (abs(val) + 0.1)
        lo = val - spread if lo is None else lo
        hi = val + spread if hi is None else hi
        sliders[name] = widgets.FloatSlider(
            value=val, min=lo, max=hi, step=(hi - lo) / 200,
            description=name, style={'description_width': '100px'},
        )
    return sliders

## 🎛 Free Play

Pick a **family**, adjust the drift (μ), volatility (σ), jump probability, and the family's own jump-size parameters. Hit **Simulate!** for a fresh random path -- press it again and you'll get a *different* path every time (check **🔒 fix seed** if you want to reproduce the exact same one while you tweak a single slider). We'll also fit the model back to the simulated data via Maximum Likelihood, so you can see how well it recovers the parameters you picked.

In [ ]:
family_dropdown = widgets.Dropdown(
    options=list(DISTRIBUTIONS.keys()), value='Skew-Normal', description='Family',
)
mu_slider = widgets.FloatSlider(value=0.1, min=-0.5, max=0.5, step=0.01, description='μ')
sigma_slider = widgets.FloatSlider(value=0.2, min=0.01, max=1.0, step=0.01, description='σ')
jump_prob_slider = widgets.FloatSlider(value=0.1, min=0.0, max=0.5, step=0.01, description='jump_prob')
fix_seed_checkbox = widgets.Checkbox(value=False, description='🔒 fix seed')
seed_input = widgets.IntText(value=0, description='seed')
simulate_button = widgets.Button(description='🎲 Simulate!', button_style='success')
free_play_output = widgets.Output()

current_jump_sliders = make_family_sliders(DISTRIBUTIONS[family_dropdown.value])
jump_sliders_box = widgets.VBox(list(current_jump_sliders.values()))


def _on_family_change(change):
    global current_jump_sliders
    current_jump_sliders = make_family_sliders(DISTRIBUTIONS[change['new']])
    jump_sliders_box.children = list(current_jump_sliders.values())


family_dropdown.observe(_on_family_change, names='value')


def run_simulation(_=None):
    with free_play_output:
        clear_output(wait=True)
        dist = DISTRIBUTIONS[family_dropdown.value]
        jump_params = {name: s.value for name, s in current_jump_sliders.items()}
        seed = seed_input.value if fix_seed_checkbox.value else int(np.random.randint(0, 2**31 - 1))

        sim = JumpDiffusionSimulator(
            mu=mu_slider.value, sigma=sigma_slider.value, jump_prob=jump_prob_slider.value,
            jump_distribution=dist, **jump_params,
        )
        times, path, _ = sim.simulate_path(T=1.0, n_steps=252, x0=0.0, seed=seed)
        sim.plot_simulation()

        increments = np.diff(path)
        dt = times[1] - times[0]
        estimator = JumpDiffusionEstimator(increments, dt, jump_distribution=dist)
        result = estimator.estimate()

        true_params = {
            'mu': mu_slider.value, 'sigma': sigma_slider.value,
            'jump_prob': jump_prob_slider.value, **jump_params,
        }
        seed_note = '' if fix_seed_checkbox.value else "  (random -- check '🔒 fix seed' to reproduce)"
        print(f'Seed used: {seed}{seed_note}\n')
        print(f"{'parameter':<16}{'true':>10}{'MLE estimate':>16}")
        for name, true_val in true_params.items():
            est_val = result['parameters'].get(name, float('nan'))
            print(f'{name:<16}{true_val:>10.4f}{est_val:>16.4f}')
        if not result['convergence']:
            print('\n⚠️  optimizer did not converge -- estimate may be unreliable')


simulate_button.on_click(run_simulation)
controls = widgets.VBox([
    family_dropdown, mu_slider, sigma_slider, jump_prob_slider, jump_sliders_box,
    widgets.HBox([fix_seed_checkbox, seed_input]), simulate_button,
])
display(widgets.HBox([controls, free_play_output]))
run_simulation()

## 🎯 Guess the Parameters

Time to flip it around. Click **🕵️ New mystery path** to simulate a path with a *randomly hidden* distribution family and parameters -- you only get to see the path, no jump markers, no numbers. Pick your best-guess family and parameters below, then hit **🔍 Reveal & Score** to see how close you got -- and how close Maximum Likelihood itself gets, for comparison.

In [ ]:
mystery = {}
game_output = widgets.Output()


def _random_true_setup():
    family_name = np.random.choice(list(DISTRIBUTIONS.keys()))
    dist = DISTRIBUTIONS[family_name]
    jump_sliders = make_family_sliders(dist)
    jump_params = {
        name: float(np.random.uniform(s.min, s.max)) for name, s in jump_sliders.items()
    }
    true_params = {
        'mu': float(np.random.uniform(-0.3, 0.3)),
        'sigma': float(np.random.uniform(0.05, 0.6)),
        'jump_prob': float(np.random.uniform(0.02, 0.3)),
        **jump_params,
    }
    seed = int(np.random.randint(0, 2**31 - 1))
    return family_name, dist, true_params, seed


def new_mystery_path(_=None):
    family_name, dist, true_params, seed = _random_true_setup()
    sim = JumpDiffusionSimulator(jump_distribution=dist, **true_params)
    times, path, _ = sim.simulate_path(T=1.0, n_steps=252, x0=0.0, seed=seed)

    mystery.update(
        family_name=family_name, dist=dist, true_params=true_params,
        seed=seed, times=times, path=path,
    )

    with game_output:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.plot(times, path, color='steelblue', label='mystery path')
        ax.set_xlabel('time'); ax.set_ylabel('value'); ax.legend()
        ax.set_title('🕵️ Which distribution & parameters generated this path?')
        ax.grid(True, alpha=0.3)
        plt.show()
        print("Pick a family + adjust the sliders below, then hit '🔍 Reveal & Score'.")


new_mystery_button = widgets.Button(description='🕵️ New mystery path', button_style='info')
new_mystery_button.on_click(new_mystery_path)
display(widgets.VBox([new_mystery_button, game_output]))
new_mystery_path()

In [ ]:
guess_family_dropdown = widgets.Dropdown(
    options=list(DISTRIBUTIONS.keys()), value='Skew-Normal', description='Your guess',
)
guess_mu_slider = widgets.FloatSlider(value=0.0, min=-0.3, max=0.3, step=0.01, description='μ')
guess_sigma_slider = widgets.FloatSlider(value=0.2, min=0.05, max=0.6, step=0.01, description='σ')
guess_jump_prob_slider = widgets.FloatSlider(value=0.1, min=0.02, max=0.3, step=0.01, description='jump_prob')

current_guess_jump_sliders = make_family_sliders(DISTRIBUTIONS[guess_family_dropdown.value])
guess_jump_sliders_box = widgets.VBox(list(current_guess_jump_sliders.values()))


def _on_guess_family_change(change):
    global current_guess_jump_sliders
    current_guess_jump_sliders = make_family_sliders(DISTRIBUTIONS[change['new']])
    guess_jump_sliders_box.children = list(current_guess_jump_sliders.values())


guess_family_dropdown.observe(_on_guess_family_change, names='value')
display(widgets.VBox([
    guess_family_dropdown, guess_mu_slider, guess_sigma_slider,
    guess_jump_prob_slider, guess_jump_sliders_box,
]))

In [ ]:
def reveal_and_score(_=None):
    with game_output:
        clear_output(wait=True)
        if not mystery:
            print("Click '🕵️ New mystery path' first!")
            return

        guess_family = guess_family_dropdown.value
        guess_jump_params = {name: s.value for name, s in current_guess_jump_sliders.items()}
        guess_params = {
            'mu': guess_mu_slider.value, 'sigma': guess_sigma_slider.value,
            'jump_prob': guess_jump_prob_slider.value, **guess_jump_params,
        }

        true_family = mystery['family_name']
        true_params = mystery['true_params']
        seed, times, path = mystery['seed'], mystery['times'], mystery['path']

        # Re-simulate the guess with the SAME seed, for a visual overlay.
        guess_dist = DISTRIBUTIONS[guess_family]
        guess_sim = JumpDiffusionSimulator(jump_distribution=guess_dist, **guess_params)
        _, guess_path, _ = guess_sim.simulate_path(
            T=1.0, n_steps=len(path) - 1, x0=0.0, seed=seed,
        )

        fig, ax = plt.subplots(figsize=(9, 4))
        ax.plot(times, path, color='steelblue', label=f'mystery path ({true_family})')
        ax.plot(times, guess_path, color='darkorange', linestyle='--', label=f'your guess ({guess_family})')
        ax.legend(); ax.set_xlabel('time'); ax.set_ylabel('value'); ax.grid(True, alpha=0.3)
        plt.show()

        if guess_family != true_family:
            print(f'❌ Wrong family! It was actually: {true_family}\n')
            print('True parameters:')
            for k, v in true_params.items():
                print(f'  {k}: {v:.4f}')
            return

        # MLE oracle: fit the TRUE family to the mystery path, for comparison.
        increments = np.diff(path)
        dt = times[1] - times[0]
        estimator = JumpDiffusionEstimator(increments, dt, jump_distribution=DISTRIBUTIONS[true_family])
        mle_params = estimator.estimate()['parameters']

        print(f'✅ Right family: {true_family}\n')
        print(f"{'parameter':<16}{'true':>10}{'your guess':>12}{'MLE estimate':>14}")
        closeness_scores = []
        for name, true_val in true_params.items():
            guess_val = guess_params.get(name, float('nan'))
            mle_val = mle_params.get(name, float('nan'))
            print(f'{name:<16}{true_val:>10.4f}{guess_val:>12.4f}{mle_val:>14.4f}')
            rel_err = abs(guess_val - true_val) / (abs(true_val) + 1e-6)
            closeness_scores.append(max(0.0, 1 - rel_err))

        score = 100 * np.mean(closeness_scores)
        if score >= 90:
            tier = '🏆 Amazing!'
        elif score >= 70:
            tier = '👍 Pretty good!'
        else:
            tier = '🤔 Room to improve...'
        print(f'\nCloseness score: {score:.0f}/100 -- {tier}')


reveal_button = widgets.Button(description='🔍 Reveal & Score', button_style='warning')
reveal_button.on_click(reveal_and_score)
display(reveal_button)
reveal_and_score()

---
➡️ Want to see this applied to real market data instead of simulations? Check out [`sp500_jump_diffusion_example_en.ipynb`](https://colab.research.google.com/github/jdospina/jump-diffusion-estimation/blob/main/notebooks/sp500_jump_diffusion_example_en.ipynb) (opens in Colab).

## Standard errors via likelihood profiling

Once we have estimated the parameters (preferably using Differential Evolution to secure the global optimum), we can compute standard errors and 95% confidence intervals using **likelihood profiling**.

This technique is far more robust than numerically computing the Hessian matrix (which is often unstable for complex mixtures).

In [ ]:
# Estimate standard errors by profiling (n_points=5 for speed here).
# Make sure you have run estimate() on the estimator first.
try:
    # Try estimator_de if it exists (from the Differential Evolution showcase)
    est = estimator_de
except NameError:
    try:
        # Try estimator if it exists (from the general example)
        est = estimator
    except NameError:
        print("Make sure to define an 'estimator' and run estimate() first.")
        est = None

if est is not None:
    print("Computing standard errors...")
    se_results = est.estimate_standard_errors(n_points=5, confidence_level=0.95)

    # Print the updated diagnostics, now including the SE / interval table
    est.diagnostics()

    # Plot the profile likelihood curves for each parameter
    est.plot_profiles()